# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** Ana Sofía Eggenberger

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [47]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [48]:
penguins.shape

penguins.dtypes

penguins.isna().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [49]:
limpio = penguins.dropna()

filas_perdidas = penguins.shape[0] - limpio.shape[0]
print(filas_perdidas)

11


### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [50]:
pesados_biscoe = limpio[
    (limpio["island"] == "Biscoe") &
    (limpio["body_mass_g"] > 4500)
]

pesados_biscoe = pesados_biscoe.sort_values("body_mass_g", ascending=False)

pesados_biscoe

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
237,Gentoo,Biscoe,49.2,15.2,221.0,6300.0,MALE
253,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,MALE
337,Gentoo,Biscoe,48.8,16.2,222.0,6000.0,MALE
297,Gentoo,Biscoe,51.1,16.3,220.0,6000.0,MALE
299,Gentoo,Biscoe,45.2,16.4,223.0,5950.0,MALE
...,...,...,...,...,...,...,...
248,Gentoo,Biscoe,48.2,14.3,210.0,4600.0,FEMALE
306,Gentoo,Biscoe,43.4,14.4,218.0,4600.0,FEMALE
296,Gentoo,Biscoe,47.5,14.2,209.0,4600.0,FEMALE
328,Gentoo,Biscoe,43.3,14.0,208.0,4575.0,FEMALE


### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [51]:
resumen = limpio.groupby(["species", "sex"]).agg({
    "body_mass_g": ["mean", "count"]
})

resumen

body_mass_g      
                         mean count
species   sex                      
Adelie    FEMALE  3368.835616    73
          MALE    4043.493151    73
Chinstrap FEMALE  3527.205882    34
          MALE    3938.970588    34
Gentoo    FEMALE  4679.741379    58
          MALE    5484.836066    61

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [52]:
limpio["proporcion_pico"] = limpio["bill_length_mm"] / limpio["bill_depth_mm"]

limpio.groupby("species")["proporcion_pico"].mean()

/tmp/ipykernel_2865/1657097181.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  limpio["proporcion_pico"] = limpio["bill_length_mm"] / limpio["bill_depth_mm"]


,proporcion_pico
species,
Adelie,2.121478
Chinstrap,2.653756
Gentoo,3.176602


### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [53]:
con_nombres = limpio.merge(
    especies,
    on="species",
    how="left"
)

con_nombres

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,proporcion_pico,nombre_cientifico
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE,2.090909,Pygoscelis adeliae
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE,2.270115,Pygoscelis adeliae
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE,2.238889,Pygoscelis adeliae
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE,1.901554,Pygoscelis adeliae
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE,1.907767,Pygoscelis adeliae
...,...,...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE,3.445255,Pygoscelis papua
329,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE,3.272727,Pygoscelis papua
330,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE,3.210191,Pygoscelis papua
331,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE,3.054054,Pygoscelis papua


## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [54]:
duckdb.sql("""
SELECT species, island, body_mass_g
FROM limpio
WHERE island = 'Biscoe'
AND body_mass_g > 4500
ORDER BY body_mass_g DESC
""").df()

,species,island,body_mass_g
0,Gentoo,Biscoe,6300.0
1,Gentoo,Biscoe,6050.0
2,Gentoo,Biscoe,6000.0
3,Gentoo,Biscoe,6000.0
4,Gentoo,Biscoe,5950.0
...,...,...,...
101,Gentoo,Biscoe,4600.0
102,Gentoo,Biscoe,4600.0
103,Gentoo,Biscoe,4600.0
104,Gentoo,Biscoe,4575.0


### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [55]:
duckdb.sql("""
SELECT species,
ROUND(AVG(body_mass_g), 0) AS promedio
FROM limpio
GROUP BY species
HAVING AVG(body_mass_g) > 4000
""").df()

,species,promedio
0,Gentoo,5092.0


### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [56]:
duckdb.sql("""
SELECT
    limpio.species,
    nombre_cientifico,
    AVG(body_mass_g) AS promedio
FROM limpio
JOIN especies
ON limpio.species = especies.species
GROUP BY limpio.species, nombre_cientifico
""").df()

,species,nombre_cientifico,promedio
0,Adelie,Pygoscelis adeliae,3706.164384
1,Gentoo,Pygoscelis papua,5092.436975
2,Chinstrap,Pygoscelis antarcticus,3733.088235


### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [57]:
duckdb.sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (
               PARTITION BY species
               ORDER BY body_mass_g DESC
           ) AS ranking
    FROM limpio
)
WHERE ranking <= 3
""").df()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,proporcion_pico,ranking
0,Gentoo,Biscoe,49.2,15.2,221.0,6300.0,MALE,3.236842,1
1,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,MALE,3.505882,2
2,Gentoo,Biscoe,51.1,16.3,220.0,6000.0,MALE,3.134969,3
3,Gentoo,Biscoe,48.8,16.2,222.0,6000.0,MALE,3.012346,3
4,Adelie,Biscoe,43.2,19.0,197.0,4775.0,MALE,2.273684,1
5,Adelie,Biscoe,41.0,20.0,203.0,4725.0,MALE,2.050000,2
6,Adelie,Torgersen,42.9,17.6,196.0,4700.0,MALE,2.437500,3
7,Chinstrap,Dream,52.0,20.7,210.0,4800.0,MALE,2.512077,1
8,Chinstrap,Dream,52.8,20.0,205.0,4550.0,MALE,2.640000,2
9,Chinstrap,Dream,53.5,19.9,205.0,4500.0,MALE,2.688442,3


## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.


**Respuesta:** En esta hoja me pareció que Pandas fue más natural para limpiar los datos con dropna() y crear una columna nueva. En cambio SQL fue mejor para hacer consultas, como el JOIN entre tablas y las agrupaciones con GROUP BY y HAVING. Después de hacer los mismos análisis en ambas herramientas, siento que Pandas es más cómodo para transformar datos y SQL para consultarlos y resumirlos.

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- Ejercicio 9

Usé IA para entender mejor cómo funciona un JOIN en SQL junto con GROUP BY para combinar dos tablas y calcular un promedio por especie.

Ejercicio 10

Usé IA para entender el concepto de las window functions, especialmente cómo funciona RANK() con PARTITION BY y por qué se usa una subconsulta para filtrar el ranking.

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [58]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [59]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    return (x - x.mean()) / x.std()

z = z_score(alturas)

# Verificación (descomentar):
print(round(z.mean(), 4), round(z.std(), 4))

-0.0 1.0


### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [60]:
puntos = rng.normal(size=(500, 2))
centro = np.array([1.0, 1.0])

# ¿Qué va aquí?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego elevar al cuadrado, sumar con axis=1, sacar raíz

distancias = np.sqrt(((puntos - centro) ** 2).sum(axis=1))

# ¿Cuántos puntos están a menos de 1 del centro?

cercanos = (distancias < 1).sum()

# Verificación (descomentar):
print(distancias.shape)   #(500,)
print(cercanos)

(500,)
86


### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [61]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,sex,smoker,day,time,size,pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [62]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

resumen = tips.groupby("day").agg({
    "pct": "mean",
    "time": "first",
    "total_bill": "count"
})


# Verificacion: resumen debe tener 4 filas
print(resumen.shape)

(4, 3)
